# 00 — Build the RAG dataset (retrieval corpus)

Thin wrapper around **`sample_generation.get_rag_corpus()`**. The corpus build now lives in the centralized `sample_generation.py` util, alongside the other batches (`get_tuning_sample` / `get_robustness_batch` / `get_test_batch`) — not in `rag_utils`.

It creates **`data/processed/rag_corpus.csv`** — the pool of labelled *precedent* loans the 05_rag notebooks retrieve from: the full 2012–2014 frame with every evaluation batch removed, so no eval (or test) loan can ever be retrieved.

- **Full corpus** when the raw `data/raw/accepted_2007_to_2018Q4.csv.gz` is present.
- **Dev fallback** (~100-row `tuning_sample`, disjoint from `robustness_batch`) when it is absent — lets Phase 5 run before the raw file is added.

The `robustness_batch` (eval set) and held-out `test_batch` are always excluded, and zero overlap with the eval set is asserted on every build.

You don't strictly need this notebook: `python sample_generation.py` builds every batch including the corpus, and 05a/b/c call `get_rag_corpus()` lazily (load-if-exists). It's here as the documented Phase-5 data step.

In [ ]:
import sys; sys.path.insert(0, '..')
from sample_generation import get_rag_corpus, get_robustness_batch
import rag_utils as R   # assert_no_leakage

In [ ]:
# force=True regenerates. With the raw .csv.gz present this builds the full large
# corpus; without it, the ~100-row tuning_sample dev fallback. Plain get_rag_corpus()
# (no force) is load-if-exists and returns the committed file.
corpus = get_rag_corpus(force=True)

# Leakage guard: the corpus must be disjoint from the evaluation set (robustness_batch).
robustness = get_robustness_batch()
R.assert_no_leakage(corpus, robustness)
print(f'RAG corpus rows       : {len(corpus)}')
print(f'Robustness (eval) rows: {len(robustness)}')
print('Leakage check         : PASSED (corpus ∩ robustness = ∅)')

In [ ]:
# Quick profile of the corpus
co = int((corpus['loan_status'] == 0).sum())
print(f'Charged Off: {co}  |  Fully Paid: {len(corpus) - co}')
corpus.head()